# 💬 Module 8: RAG AI Assistant (FAISS Vector Store & Sentence Transformers)

In this notebook, we construct a product and analytics knowledge base from retail metrics, encode text into vector embeddings using **Sentence Transformers (`all-MiniLM-L6-v2`)**, build a **FAISS Vector Database Index**, and query natural language business answers using context retrieval and **Flan-T5**.

---

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

cleaned_path = os.path.join("..", "data", "processed", "cleaned_retail_data.csv")
df = pd.read_csv(cleaned_path)
print(f"Loaded Cleaned Data: {df.shape[0]:,} rows")

### Step 1: Construct Knowledge Base Documents

In [ ]:
total_rev = df['TotalPrice'].sum()
total_orders = df['InvoiceNo'].nunique()
total_cust = df['CustomerID'].nunique()
total_prod = df['StockCode'].nunique()

top_prods = df.groupby('Description')['TotalPrice'].sum().sort_values(ascending=False).head(10)
top_prod_str = ", ".join([f"{p} (£{r:,.2f})" for p, r in top_prods.items()])

docs = [
    {"topic": "summary", "content": f"The Online Retail Platform achieved total revenue of £{total_rev:,.2f} across {total_orders:,} orders, {total_cust:,} customers, and {total_prod:,} products."},
    {"topic": "top_products", "content": f"The top revenue products are: {top_prod_str}. Single highest item: '{top_prods.index[0]}' (£{top_prods.iloc[0]:,.2f})."}
]

for doc in docs:
    print(f"Topic: {doc['topic']}")
    print(f"Content: {doc['content']}\n")

### Step 2: Generate Sentence Embeddings (`all-MiniLM-L6-v2`)

In [ ]:
st_model = SentenceTransformer('all-MiniLM-L6-v2')
texts = [doc['content'] for doc in docs]
embeddings = st_model.encode(texts, convert_to_numpy=True)
print(f"Generated Embeddings Matrix Shape: {embeddings.shape}")

### Step 3: Build & Save FAISS Vector Database

In [ ]:
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings.astype('float32'))

v_dir = os.path.join("..", "vector_db")
os.makedirs(v_dir, exist_ok=True)
faiss.write_index(index, os.path.join(v_dir, "faiss_index.bin"))
with open(os.path.join(v_dir, "product_metadata.pkl"), 'wb') as f:
    pickle.dump(docs, f)

print("FAISS vector store built and saved successfully!")

### Step 4: Semantic Query Test

In [ ]:
query_text = "Which products generated highest revenue?"
q_emb = st_model.encode([query_text], convert_to_numpy=True)
distances, indices = index.search(q_emb.astype('float32'), k=1)

retrieved_content = docs[indices[0][0]]['content']
print(f"❓ Question: '{query_text}'")
print(f"💡 Retrieved Context:\n{retrieved_content}")